# Executive Summary – Local Test Notebook

**Standalone** – contains identical logic to `scripts/executive_summary.py` but does not
import or reference it. Both files are maintained independently.

**Environment detection (same as the .py file)**
- Uses `is_running_in_airflow()` which checks both:
  1. Whether Airflow can be imported (`_HAS_AIRFLOW`)
  2. Whether `AIRFLOW_CTX_DAG_ID` env var is set (only true inside a real DAG run)
- In a notebook this is always `False`, so local env vars / getpass are used.

**Steps**
1. Cell 1 – install local deps (once)  
2. Cell 2 – set credentials via getpass  
3. Cell 3 – all shared helpers (identical logic to executive_summary.py)  
4. Cell 4 – choose month  
5. Cell 5 – query GP and inspect data  
6. Cell 6 – generate summary  
7. Cell 7 – save to .txt (optional)

In [ ]:
# ── Cell 1: Install local dependencies (run once if needed) ────────────────────
# !pip install psycopg2-binary openai pandas

In [ ]:
# ── Cell 2: Credentials ─────────────────────────────────────────────────
#
# Set env vars BEFORE Cell 3 so the connection helpers pick them up.
# Ignored when is_running_in_airflow() is True (never the case in a notebook).

import os
import getpass

# Greenplum — host/port/db/user default to config.py values; only password is required
os.environ['GP_HOST']     = 'greenplum-rdsp.zur.swissbank.com'
os.environ['GP_PORT']     = '5432'
os.environ['GP_DB']       = 'gprdsp'
os.environ['GP_USER']     = 'ds_rdsp_dev'
os.environ['GP_SCHEMA']   = 'core_ikg'
os.environ['GP_PASSWORD'] = getpass.getpass('Greenplum password: ')

os.environ['OPENAI_API_KEY']  = getpass.getpass('OpenAI / Azure API key: ')
os.environ['OPENAI_BASE_URL'] = 'https://cirruspl-staat-ste-dev-ai.openai.azure.com/openai/v1/'

print('Credentials stored in environment variables.')

In [ ]:
# ── Cell 3: All shared helpers (identical logic to scripts/executive_summary.py) ──

from __future__ import annotations

import os
import logging
from datetime import datetime
from typing import Optional

import pandas as pd
from openai import OpenAI


# ── Config constants (mirrors config.py) ──────────────────────────────────────

GREENPLUM_HOST = 'greenplum-rdsp.zur.swissbank.com'
GREENPLUM_PORT = 5432
GREENPLUM_DB   = 'gprdsp'
GREENPLUM_USER = 'ds_rdsp_dev'


# ── Environment detection ──────────────────────────────────────────────

try:
    from airflow.models import Connection, Variable                      # type: ignore
    from airflow.configuration import conf                               # type: ignore
    _HAS_AIRFLOW = True
except Exception:
    Variable   = None  # type: ignore
    Connection = None  # type: ignore
    _HAS_AIRFLOW = False


def is_running_in_airflow() -> bool:
    """True only when Airflow is installed AND we are inside a DAG run."""
    return _HAS_AIRFLOW and bool(os.environ.get('AIRFLOW_CTX_DAG_ID'))


print(f'Airflow available        : {_HAS_AIRFLOW}')
print(f'Running inside Airflow   : {is_running_in_airflow()}')
print('Mode:', 'PRODUCTION (Airflow)' if is_running_in_airflow() else 'LOCAL (env vars)')


# ── Model constants ──────────────────────────────────────────────────────

MODEL_NAME  = 'gpt-4.1'
MAX_TOKENS  = 12000
TEMPERATURE = 0.1

POSTGRES_CONN_ID_VAR    = 'GP_Dash_connect'
IKG_SCHEMA_VAR          = 'IKG_DASHBOARD_SCHEMA'
OPENAI_CONN_ID          = 'STAAT-DS-OPENAI-LLM'
DEFAULT_OPENAI_BASE_URL = 'https://cirruspl-staat-ste-dev-ai.openai.azure.com/openai/v1/'
DEFAULT_GP_SCHEMA       = 'core_ikg'

_client: Optional[OpenAI] = None

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# ── Connection helpers ────────────────────────────────────────────────

def _get_openai_client() -> OpenAI:
    """Production: Airflow Connection STAAT-DS-OPENAI-LLM.
    Local: env vars OPENAI_API_KEY / OPENAI_BASE_URL.
    """
    global _client
    if _client is not None:
        return _client
    if is_running_in_airflow():
        conn     = Connection.get_connection_from_secrets(OPENAI_CONN_ID)
        api_key  = conn.password
        base_url = conn.host
    else:
        api_key  = os.environ['OPENAI_API_KEY']
        base_url = os.environ.get('OPENAI_BASE_URL', DEFAULT_OPENAI_BASE_URL)
    _client = OpenAI(api_key=api_key, base_url=base_url)
    return _client


def _get_gp_conn(allow_prompt: bool = True):
    """Production: PostgresHook via Variable 'GP_Dash_connect'.
    Local: psycopg2 using GP_* env vars or getpass.
    """
    if is_running_in_airflow():
        from airflow.providers.postgres.hooks.postgres import PostgresHook  # type: ignore
        conn_id = Variable.get(POSTGRES_CONN_ID_VAR)
        return PostgresHook(postgres_conn_id=conn_id).get_conn()
    import psycopg2
    host     = os.environ.get('GP_HOST', GREENPLUM_HOST)
    port     = int(os.environ.get('GP_PORT', GREENPLUM_PORT))
    dbname   = os.environ.get('GP_DB', GREENPLUM_DB)
    user     = os.environ.get('GP_USER', GREENPLUM_USER)
    password = os.environ.get('GP_PASSWORD')
    if not password:
        if allow_prompt:
            password = getpass.getpass('Enter Greenplum password: ')
        else:
            raise RuntimeError('GP_PASSWORD env var is required for non-interactive runs.')
    return psycopg2.connect(host=host, port=port, dbname=dbname, user=user, password=password)


def _get_schema() -> str:
    if is_running_in_airflow():
        return Variable.get(IKG_SCHEMA_VAR)
    return os.environ.get('GP_SCHEMA', DEFAULT_GP_SCHEMA)


# ── SQL helpers ──────────────────────────────────────────────────────────

def _build_month_query(month_value: str, schema: str) -> str:
    staat = f'{schema}.staat_insight_release'
    odm   = f'{schema}.odm_release_details'
    return f"""
    WITH target_iterations AS (
        SELECT iteration_end_date, MAX(batch) AS max_batch
        FROM {staat}
        WHERE TO_CHAR(prod_release_date::date, 'YYYY-MM') = '{month_value}'
        GROUP BY iteration_end_date
    ),
    staat_latest AS (
        SELECT s.*
        FROM {staat} s
        INNER JOIN target_iterations ti
            ON  s.iteration_end_date = ti.iteration_end_date
            AND s.batch              = ti.max_batch
    ),
    odm_latest AS (
        SELECT o.*
        FROM {odm} o
        INNER JOIN target_iterations ti
            ON  o.iteration_end_date = ti.iteration_end_date
            AND o.batch              = ti.max_batch
    )
    SELECT
        s.id_x, s.title, s.labels, s.issue_summary, s.state, s.weight,
        s.prod_release_date, s.iteration_end_date, s.iteration_start_date, s.batch,
        o.rule_name, o.target_type, o.change_type
    FROM staat_latest s
    LEFT JOIN odm_latest o
        ON  s.id_x               = o.issue_id
        AND s.iteration_end_date = o.iteration_end_date
    ORDER BY s.iteration_end_date, s.id_x
    """


def _build_reactivated_query(insight_types: list, schema: str) -> str:
    staat     = f'{schema}.staat_insight_release'
    odm       = f'{schema}.odm_release_details'
    in_clause = ', '.join(f"'{t}'" for t in insight_types)
    return f"""
    WITH odm_ranked AS (
        SELECT o.*,
               MAX(o.batch) OVER (PARTITION BY o.iteration_end_date) AS max_batch
        FROM {odm} o
        WHERE o.rule_name IN ({in_clause})
    ),
    odm_latest AS (
        SELECT * FROM odm_ranked WHERE batch = max_batch
    )
    SELECT
        o.issue_id AS id_x, o.iteration_end_date, o.batch,
        o.rule_name, o.target_type, o.change_type,
        s.title, s.issue_summary, s.labels, s.state, s.weight,
        s.prod_release_date, s.iteration_start_date
    FROM odm_latest o
    LEFT JOIN {staat} s
        ON  o.issue_id           = s.id_x
        AND o.iteration_end_date = s.iteration_end_date
        AND o.batch              = s.batch
    ORDER BY o.rule_name, o.iteration_end_date, o.issue_id
    """


# ── Reactivated insight helpers ─────────────────────────────────────────

def get_reactivated_insights(schema: str, conn, year: int, month: int) -> pd.DataFrame:
    sql = f"""
        SELECT DISTINCT insight_type
        FROM {schema}.odm_exclusion_insight_type
        WHERE EXTRACT(YEAR  FROM last_upd_dte::date) = {year}
          AND EXTRACT(MONTH FROM last_upd_dte::date) = {month}
          AND is_curr = 0
    """
    try:
        df = pd.read_sql_query(sql, conn)
    except Exception as exc:
        logger.warning('[reactivated] exclusion table query failed: %s', exc)
        return pd.DataFrame()
    if df.empty:
        logger.info('[reactivated] No reactivated insight types found.')
        return pd.DataFrame()
    insight_types = df['insight_type'].dropna().str.strip().tolist()
    logger.info('[reactivated] Found %d reactivated type(s): %s', len(insight_types), insight_types)
    try:
        details = pd.read_sql_query(_build_reactivated_query(insight_types, schema), conn)
        details['change_type'] = details['change_type'].replace({'added': 'new'})
        return details
    except Exception as exc:
        logger.warning('[reactivated] story detail query failed: %s', exc)
        return pd.DataFrame()


# ── Public data helper ───────────────────────────────────────────────────

def get_exec_summary_data(month_value: str, allow_prompt: bool = True) -> tuple:
    schema   = _get_schema()
    conn     = _get_gp_conn(allow_prompt=allow_prompt)
    year, month_num = (int(x) for x in month_value.split('-'))
    current_df = pd.read_sql_query(_build_month_query(month_value, schema), conn)
    current_df['change_type'] = current_df['change_type'].replace({'added': 'new'})
    reactivated_df = get_reactivated_insights(schema, conn, year, month_num)
    try:
        next_period = str(pd.Period(month_value, 'M') + 1)
        next_df     = pd.read_sql_query(_build_month_query(next_period, schema), conn)
        next_df['change_type'] = next_df['change_type'].replace({'added': 'new'})
        next_month_df = next_df if not next_df.empty else None
    except Exception:
        next_month_df = None
    conn.close()
    return current_df, next_month_df, reactivated_df


def get_month_options_from_db() -> list:
    schema = _get_schema()
    conn   = _get_gp_conn()
    sql    = f"""
        SELECT DISTINCT TO_CHAR(prod_release_date::date, 'YYYY-MM') AS month_val
        FROM {schema}.staat_insight_release
        WHERE prod_release_date IS NOT NULL
        ORDER BY month_val DESC
    """
    df = pd.read_sql_query(sql, conn)
    conn.close()
    options = []
    for val in df['month_val']:
        try:
            dt = pd.Period(val, 'M').to_timestamp()
            options.append({'label': dt.strftime('%B %Y'), 'value': val})
        except Exception:
            pass
    return options


# ── Label helpers ──────────────────────────────────────────────────────────

def _has_label(labels_value, target: str) -> bool:
    if not labels_value or not isinstance(labels_value, str):
        return False
    return target.lower() in [lbl.strip().lower() for lbl in labels_value.split(',')]


def _filter_by_label(df: pd.DataFrame, label: str) -> pd.DataFrame:
    if df is None or df.empty or 'labels' not in df.columns:
        return pd.DataFrame()
    return df.loc[df['labels'].apply(lambda v: _has_label(v, label))]


# ── Prompt helpers ───────────────────────────────────────────────────────

def _build_reactivated_text(reactivated_df: pd.DataFrame, skip_rules: set = None) -> str:
    """Format reactivated rows; skip any rule already listed as New Insight."""
    if reactivated_df is None or reactivated_df.empty:
        return 'No reactivated insights found for this period.'
    skip = skip_rules or set()
    lines: list = []
    rule_num = 0
    for rule_name, group in reactivated_df.groupby('rule_name', sort=True):
        if rule_name in skip:
            continue
        rule_num += 1
        lines.append(f'Reactivated Insight #{rule_num}: {rule_name}')
        seen_ids: set = set()
        story_num = 0
        for _, row in group.iterrows():
            id_x = row.get('id_x', '')
            if id_x and id_x not in seen_ids:
                seen_ids.add(id_x)
                story_num += 1
                lines.append(f'  Story #{story_num}:')
                lines.append(f'    Title:   {row.get("title", "N/A")}')
                lines.append(f'    Summary: {row.get("issue_summary", "N/A")}')
        lines.append('')
    return '\n'.join(lines).strip() if lines else 'No reactivated insights found for this period.'


def _build_prompt(issues_df: pd.DataFrame, next_month_df=None, reactivated_df=None) -> str:
    # Section 1
    issues_lines: list = []
    seen_ids: set = set()
    for _, row in issues_df.iterrows():
        id_x = row.get('id_x', '')
        if id_x and id_x not in seen_ids:
            seen_ids.add(id_x)
            issues_lines.append(
                f"Story #{len(issues_lines)+1}:\n"
                f"Title:   {row.get('title','N/A')}\n"
                f"Summary: {row.get('issue_summary','N/A')}\n"
                f"State:   {row.get('state','N/A')}\n"
                f"Labels:  {row.get('labels','N/A')}\n"
                f"Weight:  {row.get('weight','N/A')}"
            )
    issues_text = '\n\n'.join(issues_lines[:20]) if issues_lines else 'No issues data available.'

    # Section 2
    top_df    = _filter_by_label(issues_df, 'Top Feature')
    top_lines: list = []
    seen_top: set = set()
    for _, row in top_df.iterrows():
        id_x = row.get('id_x', '')
        if id_x and id_x not in seen_top:
            seen_top.add(id_x)
            top_lines.append(
                f"Top Feature #{len(top_lines)+1}:\n"
                f"Title:   {row.get('title','N/A')}\n"
                f"Summary: {row.get('issue_summary','N/A')}\n"
                f"Labels:  {row.get('labels','N/A')}\n"
                f"Rule:    {row.get('rule_name','N/A')}\n"
                f"Change:  {row.get('change_type','N/A')}"
            )
    top_feature_text = ('\n\n'.join(top_lines[:5]) if top_lines
                        else "No issue labelled 'Top Feature' found for this period.")

    # Section 3
    ni_df    = _filter_by_label(issues_df, 'New Insight')
    ni_lines: list = []
    seen_rules: set = set()   # carried into reactivated block for dedup
    for _, row in ni_df.iterrows():
        rule = row.get('rule_name', '')
        if rule and rule not in seen_rules:
            seen_rules.add(rule)
            ni_lines.append(
                f"New Insight #{len(ni_lines)+1}:\n"
                f"Rule Name:   {rule}\n"
                f"Target Type: {row.get('target_type','N/A')}\n"
                f"Change Type: {row.get('change_type','N/A')}\n"
                f"Title:       {row.get('title','N/A')}\n"
                f"Summary:     {row.get('issue_summary','N/A')}"
            )
    insights_text = ('\n\n'.join(ni_lines[:15]) if ni_lines
                     else "No issues labelled 'New Insight' found for this period.")

    reactivated_text = _build_reactivated_text(reactivated_df, skip_rules=seen_rules)

    # Section 6
    next_lines: list = []
    if next_month_df is not None and not next_month_df.empty:
        next_ni_df = _filter_by_label(next_month_df, 'New Insight')
        next_seen: set = set()
        for _, row in next_ni_df.iterrows():
            rule  = row.get('rule_name', '')
            title = row.get('title', '')
            key   = rule or title
            if key and key not in next_seen:
                next_seen.add(key)
                next_lines.append(f"- {row.get('title', rule)}")
    next_month_text = ('\n'.join(next_lines[:10]) if next_lines
                       else 'No next-month data found \u2013 provide directional focus areas based on current trends.')

    return f"""
    You are an expert product manager and technical writer. Based on the following GitLab issues and
    new insights data, generate a comprehensive executive summary report.

    === GITLAB ISSUES DATA (all stories for this month) ===
    {issues_text}

    === TOP FEATURE DATA (issues labelled \"Top Feature\") ===
    {top_feature_text}

    === NEW INSIGHTS DATA (issues labelled \"New Insight\") ===
    {insights_text}

    === REACTIVATED INSIGHTS (insight types re-enabled this month) ===
    These insights were previously excluded but have been re-activated this month.
    Include them as additional items in the New Insights section (Section 3), clearly noting they are reactivated.
    {reactivated_text}

    === REQUIRED OUTPUT FORMAT ===
    Please analyse the data and provide a structured report. Use \"<Question>: <answer>\" style:

    1. Executive Summary:
    - What materially changed this month: <answer>
    - Why it matters to the business: <answer>

    2. Top Feature (or Insight) of the Month:
    - Insight name: <answer>
    - Brief description / example of narrative: <answer>
    - Main benefit / value: <answer>

    3. New Insights (ranked by importance/impact, including reactivated insights):
    For each new or reactivated insight:
    - Insight name: <answer>
    - Brief description / example of narrative: <answer>
    - Main benefit / value: <answer>
    - Status: New | Reactivated

    4. Process Improvements & Optimization:
    - Name / brief description: <answer>
    - How did we do it? (If AI was used, altered process, etc.): <answer>
    - Benefits: <answer>

    5. Platform Maintenance & Stability:
    - Maintenance, fixes, or technical improvements: <answer>
    - Why this matters (risk reduction, performance, cost control): <answer>

    6. New Insights \u2013 Coming Soon:
    Use the following next-month data (if available):
    {next_month_text}
    - Focus areas (2-4 items max): list Insight Title only, limited to \"New Insight\" labelled items

    === INSTRUCTIONS ===
    - Be concise and business-focused
    - Rank insights by business impact and value
    - Use clear, non-technical language where possible
    - Focus on outcomes and benefits, not just features
    - If certain sections have no relevant data, state \"No significant changes in this area\"
    - Ensure all answers are data-driven based on the provided information
    - Section 1 Executive Summary: 2 sentences per question
    - Section 2 Top Feature: pick the single most impactful item from the \"Top Feature\" labelled issues
    - Section 3 New Insights: list each insight on a separate line; include reactivated insights at the end marked as \"Reactivated\". Bundle multiple stories for the same rule under one entry
    - Section 6 New Insights \u2013 Coming Soon: simple list of Insight Titles only. If no next-month data, provide directional focus areas
    - Do NOT include instruction notes in the final output
    """.strip()


# ── LLM call ─────────────────────────────────────────────────────────────

def generate_summary(issues_df: pd.DataFrame, next_month_df=None, reactivated_df=None) -> str:
    client   = _get_openai_client()
    prompt   = _build_prompt(issues_df, next_month_df=next_month_df, reactivated_df=reactivated_df)
    response = client.chat.completions.create(
        model       =MODEL_NAME,
        messages    =[{'role': 'user', 'content': prompt}],
        max_tokens  =MAX_TOKENS,
        temperature =TEMPERATURE,
    )
    return response.choices[0].message.content.strip()


def build_save_content(summary_text: str, month_label: str) -> str:
    header = (
        f'Executive Summary Report \u2013 {month_label}\n'
        f'Generated on: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
        + '=' * 80 + '\n\n'
    )
    return header + summary_text


print('All helpers loaded successfully.')

In [ ]:
# ── Cell 4: Choose a month ───────────────────────────────────────────────
#
# When run from the DAG: month = datetime.now() (handled automatically).
# When run locally (here): prompt the user.

month_options = get_month_options_from_db()

print('Available months (latest first):')
for i, opt in enumerate(month_options, start=1):
    print(f"  {i:>2}. {opt['label']}  ({opt['value']})")

print()
raw = input("Enter the month to summarise (e.g. 'May 2026', 'May-2026', or '2026-05'): ").strip()

try:
    SELECTED_MONTH = str(pd.Period(raw, 'M'))
except Exception:
    cleaned = raw.replace('-', ' ').replace('/', ' ')
    SELECTED_MONTH = datetime.strptime(cleaned, '%B %Y').strftime('%Y-%m')

SELECTED_LABEL = pd.Period(SELECTED_MONTH, 'M').to_timestamp().strftime('%B %Y')
print(f'\nSelected: {SELECTED_LABEL}  [{SELECTED_MONTH}]')

In [ ]:
# ── Cell 5: Query GP and inspect data ───────────────────────────────────

df_current, df_next_month, df_reactivated = get_exec_summary_data(SELECTED_MONTH)

NEXT_LABEL = (pd.Period(SELECTED_MONTH, 'M') + 1).to_timestamp().strftime('%B %Y')
print(f'Current month ({SELECTED_LABEL}) rows : {len(df_current)}')
print(f'Next month    ({NEXT_LABEL}) rows : {len(df_next_month) if df_next_month is not None else 0}')
print(f'Reactivated types                 : {0 if df_reactivated.empty else df_reactivated["rule_name"].nunique()}')

print('\n── Current month data (first 10 rows) ──')
display(df_current.head(10))

print('\n── Label breakdown ──')
display(
    df_current['labels'].dropna().str.split(',').explode()
    .str.strip().value_counts()
)

print("\n── Rows labelled 'Top Feature' ──")
display(
    df_current[
        df_current['labels'].fillna('').apply(lambda v: _has_label(v, 'Top Feature'))
    ][['id_x','title','labels','rule_name','change_type']]
)

print("\n── Rows labelled 'New Insight' ──")
display(
    df_current[
        df_current['labels'].fillna('').apply(lambda v: _has_label(v, 'New Insight'))
    ][['id_x','title','labels','rule_name','change_type']]
)

print('\n── Reactivated insights (bundled by rule_name) ──')
if df_reactivated.empty:
    print('  (none found for this month)')
else:
    display(df_reactivated[['rule_name','id_x','title','issue_summary','iteration_end_date','batch']])
    print('\nFormatted prompt block:')
    print(_build_reactivated_text(df_reactivated))

In [ ]:
# ── Cell 6: Generate the executive summary ───────────────────────────────

print(f'Calling {MODEL_NAME} \u2026 this may take up to 30 seconds.')

SUMMARY_TEXT = generate_summary(
    df_current,
    next_month_df  =df_next_month,
    reactivated_df =df_reactivated if not df_reactivated.empty else None,
)

print('\n' + '=' * 80)
print(f'Executive Summary \u2013 {SELECTED_LABEL}')
print('=' * 80 + '\n')
print(SUMMARY_TEXT)

In [ ]:
# ── Cell 7: Save to .txt file (optional) ────────────────────────────────

content  = build_save_content(SUMMARY_TEXT, SELECTED_LABEL)
filename = f'executive_summary_{SELECTED_MONTH}.txt'

with open(filename, 'w', encoding='utf-8') as f:
    f.write(content)

print(f'Saved \u2192 {filename}')